# SettleAI backend on Colab's free GPU

Runs the full FastAPI backend (`api.py`: ASR, LLM, RAG, TTS) here on a Colab GPU runtime, then exposes it through a public tunnel URL. Your **frontend keeps running locally** on your own machine — you just point it at the tunnel URL instead of `localhost:8000`.

**Before you run this:**
1. Runtime → Change runtime type → GPU (T4 is fine).
2. Add your Groq key as a Colab secret: click the key icon 🔑 in the left sidebar → add secret named `GROQ_API_KEY` → toggle notebook access on. (Falls back to a manual prompt if you skip this.)
3. If your repo is private, you'll need a GitHub token for the clone step — see that cell.

**Heads up — this is ephemeral.** Everything here (Ollama models, the Chroma vector store, the tunnel) lives only as long as this Colab session. If it disconnects, you lose the RAG-ingested data unless you re-ingest or persist `chroma_db/` to Drive yourself.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU attached — go to Runtime > Change runtime type > GPU'

## 1. Clone the repo
Defaults to `colab-gpu-backend`, which has everything this notebook needs already committed (today's TTS latency/description fixes + this notebook itself). Change `BRANCH` if you want a different one.

In [ ]:
BRANCH = "colab-gpu-backend"
REPO_URL = "https://github.com/cive202/settleAI-NepaliVoiceAssistance.git"

!git clone -b {BRANCH} {REPO_URL} repo
%cd repo

## 1b. (Optional) Upload local edits not yet pushed
If you have uncommitted changes on your machine (e.g. `config.py`, `tts.py`, `api.py`) that aren't in the GitHub branch yet, upload them here to overwrite the cloned copies. Skip this cell if your branch is already up to date.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select config.py / tts.py / api.py etc. from your local machine
for name in uploaded:
    print(f"overwrote {name}")

## 2. Install Python dependencies
`parler-tts` isn't on PyPI, so it's installed separately (see `requirements_local.txt`'s own note on this). This pins `transformers==4.46.1`, which is expected.

In [ ]:
!pip install -q -r requirements_local.txt
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q playwright
!playwright install --with-deps chromium

# requirements_local.txt installs CPU-only onnxruntime by default (for
# asr/indic_conformer.py); swap in the GPU build since this notebook runs
# on a CUDA box.
!pip install -q -U onnxruntime-gpu==1.20.1

# Colab's base image ships an old protobuf (for TensorFlow compat) that's
# missing google.protobuf.runtime_version; chromadb/sentence-transformers
# (pulled in via langchain-chroma/langchain-text-splitters) need a newer
# one. The three installs above don't jointly resolve dependencies, so
# force a compatible protobuf last. api.py runs as its own subprocess
# later (not imported into this kernel), so no kernel restart is needed
# for it to pick this up.
#
# Pinned to <6 rather than left open: an unbounded ">=5.28.0" resolves to
# whatever's newest (7.x at time of writing), which conflicts with more
# of Colab's preinstalled stack (grpcio-status, google-ai-generativelanguage,
# ydf all cap below 6-7) than the 5.28-5.x window does. descript-audiotools
# wanting protobuf<5.0 is a separate, unavoidable conflict with the
# transformers/chromadb chain above — pip will warn about it, but it isn't
# actually imported on this project's inference-only code path.
!pip install -q -U "protobuf>=5.28.0,<6"

## 3. Install & start Ollama (needed by rag/service.py + rag/store.py)
Pulls `llama3.1` (RAG answer generation) and `nomic-embed-text` (RAG embeddings). This step is the slowest one — llama3.1 is a multi-GB download.

In [ ]:
!apt-get -qq install -y zstd  # the ollama installer needs this to extract itself; not preinstalled on Colab
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)  # give the daemon a moment to bind before pulling

!ollama pull llama3.1
!ollama pull nomic-embed-text

# Pulling only downloads the weights — Ollama still loads a model from disk
# into memory on its first request, which for an 8B model can alone exceed
# RAG_OLLAMA_TIMEOUT_S (60s in config.py). Force that load to happen now,
# outside the timeout window, so the first real user request isn't the one
# that pays for it. keep_alive keeps it resident past Ollama's 5-minute
# default idle-unload, since there's usually a gap here while you set up
# the frontend before asking a real question.
import requests

print("warming up llama3.1...")
requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "llama3.1", "prompt": "hi", "stream": False, "keep_alive": "30m"},
    timeout=300,
)
print("warming up nomic-embed-text...")
requests.post(
    "http://localhost:11434/api/embeddings",
    json={"model": "nomic-embed-text", "prompt": "hi", "keep_alive": "30m"},
    timeout=300,
)

!ollama ps  # confirm both show up, and check the PROCESSOR column for GPU vs CPU

## 4. Secrets → `.env_local` / environment
Reads `GROQ_API_KEY` from Colab's secret manager if you added it (recommended); otherwise prompts for it without echoing to the notebook output.

Also needs an **HF token** — `ai4bharat/indic-conformer-600m-multilingual` (the local ASR model) is a gated repo. Before running this cell: log into huggingface.co, open the [model page](https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual), and accept its terms — I can't do that step for you. Then add a Colab secret named `HF_TOKEN` (a read-access token from huggingface.co/settings/tokens), or enter one when prompted below.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def _get_secret(name, prompt):
    value = userdata.get(name) if userdata else None
    if not value:
        value = getpass(f"{prompt}: ")
    return value


groq_key = _get_secret("GROQ_API_KEY", "GROQ_API_KEY (not stored as a Colab secret — enter manually)")
hf_token = _get_secret(
    "HF_TOKEN",
    "HF_TOKEN for the gated indic-conformer ASR model (not stored as a Colab secret — enter manually)",
)

with open(".env_local", "w") as f:
    f.write(f"GROQ_API_KEY={groq_key}\n")

# huggingface_hub reads this env var automatically for gated-repo auth —
# needs to be set before the uvicorn subprocess (which loads the ASR model)
# starts, since it inherits the environment at the point it's spawned.
os.environ["HF_TOKEN"] = hf_token

print("wrote .env_local, set HF_TOKEN")

## 5. Start the API server
Runs in the background; polls `/api/health` until model loading (`lifespan()` in `api.py`) finishes — this can take a few minutes the first time (Whisper + Parler-TTS + Ollama warm-up).

In [ ]:
import subprocess, time, requests

api_proc = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("/content/api.log", "w"),
    stderr=subprocess.STDOUT,
)

print("waiting for models to load (tail /content/api.log for progress)...")
for _ in range(180):  # up to ~15 min
    try:
        r = requests.get("http://localhost:8000/api/health", timeout=2)
        if r.ok and r.json().get("ready"):
            print("API ready.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    print("Still not ready — check /content/api.log for errors:")
    !tail -n 60 /content/api.log

## 6. Expose it publicly (Cloudflare quick tunnel — no account needed)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, re, time

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
start = time.time()
while time.time() - start < 30:
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"\nBackend is live at: {public_url}\n")
    print("On your LOCAL machine, in frontend/.env.local set:")
    print(f"  NEXT_PUBLIC_API_URL={public_url}")
    print("then restart `npm run dev` and open http://localhost:3000")
else:
    print("Couldn't find the tunnel URL in cloudflared's output — check manually.")

## 7. (Optional) sanity check from inside the notebook

In [ ]:
import requests

print(requests.get("http://localhost:8000/api/health").json())

# Uncomment to try a real round-trip once you've ingested content via /api/rag/ingest:
# resp = requests.post("http://localhost:8000/api/text", json={"text": "नमस्ते"})
# for line in resp.iter_lines():
#     print(line)

## Notes
- **RAG store is empty on a fresh clone.** `chroma_db/` isn't in git. Either mount Drive and copy a previously-persisted `chroma_db/` folder in before starting uvicorn, or call `POST /api/rag/ingest` (from this notebook or the frontend) to populate it fresh.
- **Session limits:** Colab free-tier GPU sessions disconnect after periods of inactivity or after ~12h. Treat this as a temporary dev/demo backend, not something to leave running unattended.
- **CORS:** `api.py` already allows `http://localhost:3000` / `:3001` as origins, so no backend change is needed — your frontend's origin doesn't change just because the backend moved.